# 00_create_llm_dataset

this code creates the dataset that will be run in the llm.

i uses the cosine merged that already have job ids job titles and role desc tasks skils 

it follows the monthly aproach from 1 to 14, as in the dev mode it is running a 1k per month.

In [ ]:
# fdirst create cosine merged than after llm ready dataset (second step)

In [4]:
import numpy as np
from pathlib import Path
from collections import defaultdict

# -----------------------------------------------------------------------------
# Merge (intersection on job_ids)
# -----------------------------------------------------------------------------
def merge_job_data(cos_path, llama_path, out_path):
    cos_path = Path(cos_path)
    llama_path = Path(llama_path)
    out_path = Path(out_path)

    a = np.load(cos_path, allow_pickle=True)
    b = np.load(llama_path, allow_pickle=True)

    if "job_ids" not in a.files or "job_ids" not in b.files:
        raise KeyError("Both inputs must contain 'job_ids'")

    a_ids = np.array([str(x) for x in a["job_ids"]], dtype=object)
    b_ids = np.array([str(x) for x in b["job_ids"]], dtype=object)

    b_idx_map = {jid: i for i, jid in enumerate(b_ids)}

    mask = np.array([jid in b_idx_map for jid in a_ids], dtype=bool)
    idx_a = np.where(mask)[0].astype(np.int64)
    idx_b = np.array([b_idx_map[jid] for jid in a_ids[mask]], dtype=np.int64)

    if len(idx_a) == 0:
        print(f"No matches for {out_path.name}. Skipping.")
        return 0

    merged = {}
    nA = len(a_ids)
    nB = len(b_ids)

    for k in a.files:
        v = a[k]
        if hasattr(v, "shape") and getattr(v, "ndim", 0) >= 1 and v.shape[0] == nA:
            merged[k] = v[idx_a]
        else:
            merged[k] = v

    for k in b.files:
        if k == "job_ids":
            continue
        v = b[k]
        if hasattr(v, "shape") and getattr(v, "ndim", 0) >= 1 and v.shape[0] == nB:
            merged[k] = v[idx_b]
        else:
            merged[k] = v

    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out_path, **merged)
    print(f"Saved {len(idx_a):,} rows to {out_path}")
    return len(idx_a)


# -----------------------------------------------------------------------------
# Build COS month file (Memory Optimized)
# -----------------------------------------------------------------------------
def build_month(
    model_dir,
    aspectt_npz,
    month,
    out_dir,
    alpha=0.5,
    topn=10,
    filter_job_ids=None,
    require_full_match=True,
):
    model_dir = Path(model_dir)
    out_dir = Path(out_dir)

    parts = sorted(model_dir.glob(f"{month}_shard*/part_*.npz"))
    assert parts, f"No shards found for {month} in {model_dir}"

    d0 = np.load(parts[0], allow_pickle=True)
    M = {k: [] for k in d0.files}

    # Pre-compute filter set
    want_set = set(str(x) for x in filter_job_ids) if filter_job_ids is not None else None
    total_found = 0

    for p in parts:
        d = np.load(p, allow_pickle=True)
        
        if want_set:
            # Filter at the shard level to save massive RAM overhead
            have = np.array([str(x) for x in d["job_ids"]], dtype=object)
            keep = np.array([jid in want_set for jid in have], dtype=bool)
            total_found += keep.sum()
            
            for k in M:
                v = d[k]
                if hasattr(v, "ndim") and v.ndim >= 1 and v.shape[0] == len(have):
                    M[k].append(v[keep])
                else:
                    M[k].append(v)
        else:
            for k in M:
                M[k].append(d[k])

    if want_set and require_full_match:
        if total_found < len(want_set):
            raise ValueError(
                f"{month}: missing {len(want_set) - total_found} of {len(want_set)} filter_job_ids in {model_dir}"
            )

    for k in list(M):
        arr0 = np.asarray(M[k][0])
        M[k] = arr0 if arr0.ndim == 0 else np.concatenate(M[k], axis=0)

    titles = np.load(aspectt_npz, allow_pickle=True)["titles"]

    rt = titles[M["role_topk_idx"]]
    rv = M["role_topk_val"].astype(float)
    tt = titles[M["task_topk_idx"]]
    tv = M["task_topk_val"].astype(float)

    n = rt.shape[0]
    candidates = np.empty(n, dtype=object)
    candidates_dict = np.empty(n, dtype=object)

    for i in range(n):
        agg = defaultdict(float)

        for t, w in zip(rt[i], rv[i]):
            agg[t] += alpha * float(w)

        for t, w in zip(tt[i], tv[i]):
            agg[t] += (1 - alpha) * float(w)

        candidates_dict[i] = dict(agg)
        candidates[i] = [t for t, _ in sorted(agg.items(), key=lambda x: -x[1])[:topn]]

    out_dir.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        out_dir / f"{month}.npz",
        **{k: v for k, v in M.items() if k not in ["chunk_start", "chunk_end"]},
        candidates=candidates,
        candidates_dict=candidates_dict,
        alpha=np.float32(alpha),
    )


# -----------------------------------------------------------------------------
# Driver
# -----------------------------------------------------------------------------
def build_then_merge_all_months(
    months,
    base_dir,
    model_name,
    model_dir,
    aspectt_npz,
    llama_subdir,
    alpha=0.4,
    topn=10,
):
    base = Path(base_dir)

    temp_dir = base / f"cosines/cosine_merged/{model_name}/temp"
    final_dir = base / f"cosines/cosine_merged/{model_name}"
    llama_dir = base / llama_subdir

    temp_dir.mkdir(parents=True, exist_ok=True)
    final_dir.mkdir(parents=True, exist_ok=True)

    for month in months:
        print(f"\n== {model_name} | {month} ==")

        cos_p = temp_dir / f"{month}.npz"
        llama_p = llama_dir / f"{month}.npz"
        out_p = final_dir / f"{month}.npz"

        if not llama_p.exists():
            print(f"[SKIP] Missing LLAMA: {llama_p}")
            continue

        llama = np.load(llama_p, allow_pickle=True)
        llama_ids = llama["job_ids"]

        print(f"[BUILD] {cos_p}")
        build_month(
            model_dir=model_dir,
            aspectt_npz=aspectt_npz,
            month=month,
            out_dir=temp_dir,
            alpha=alpha,
            topn=topn,
            filter_job_ids=llama_ids,
            require_full_match=False,
        )

        merge_job_data(cos_p, llama_p, out_p)


# -----------------------------------------------------------------------------
# RUN ALL MODELS
# -----------------------------------------------------------------------------
MONTHS = [f"adzuna_month{m:02d}" for m in range(1, 15)]

# CHANGED TO PROD PATHS
BASE_DIR = (
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/"
    "AISI_demo/stage_2_embeddings_and_cosines/prod"
)

ASPECTT_NPZ = (
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/"
    "AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/aspectt_vectors.npz"
)

# CHANGED LLAMA DIR TO PROD
LLAMA_PROD_SUBDIR = "llama_summary_deduplicated" # <-- Verify this matches your actual prod folder

# CHANGED E5 TO MPNET AND UPDATED TO PROD PATHS
MODELS = {
    "bge_large": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/bge_large",
}

for model_name, model_dir in MODELS.items():
    build_then_merge_all_months(
        months=MONTHS,
        base_dir=BASE_DIR,
        model_name=model_name,
        model_dir=model_dir,
        aspectt_npz=ASPECTT_NPZ,
        llama_subdir=LLAMA_PROD_SUBDIR,
        alpha=0.4,
        topn=10,
    )


== bge_large | adzuna_month01 ==
[BUILD] /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged/bge_large/temp/adzuna_month01.npz
Saved 2,657,586 rows to /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged/bge_large/adzuna_month01.npz

== bge_large | adzuna_month02 ==
[BUILD] /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged/bge_large/temp/adzuna_month02.npz
Saved 1,539,063 rows to /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged/bge_large/adzuna_month02.npz

== bge_large | adzuna_month03 ==
[BUILD] /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged/bge_large/temp/adzuna_month03.

In [ ]:
# this is second step

In [6]:
!pip install orjson


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
### TEST TEST

import numpy as np
import orjson
from pathlib import Path
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
DEV_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged"
)

JSONL_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_1_LLM_summary"
)

STAGE3_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection"
)

EMBEDS = ["bge_large"]
MONTHS = [f"adzuna_month{m:02d}" for m in range(1, 15)]
OVERWRITE = True


# ============================================================
# HELPERS
# ============================================================
def require_npz_keys(z, need, ctx=""):
    have = set(z.files)
    missing = sorted(list(set(need) - have))
    if missing:
        raise KeyError(
            f"NPZ missing keys {missing} | ctx={ctx} | have={sorted(list(have))}"
        )


def scan_jsonl_dir(jsonl_dir: Path, job_id_set: set):
    title_map = {}
    sector_map = {}
    desc_map = {}
    total_files = 0
    found = 0

    for p in sorted(jsonl_dir.glob("*.jsonl")):
        total_files += 1
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = orjson.loads(line)
                except orjson.JSONDecodeError:
                    continue

                jid = (
                    obj.get("job_id")
                    or obj.get("jobid")
                    or obj.get("id")
                    or (obj.get("meta") or {}).get("job_id")
                )
                if jid is None:
                    continue

                jid = str(jid)
                if jid not in job_id_set:
                    continue

                found += 1

                t = obj.get("job_ad_title") or obj.get("title") or obj.get("job_title") or ""
                c = obj.get("job_sector_category") or obj.get("category_name") or obj.get("category") or ""
                d = (
                    obj.get("job_description")
                    or obj.get("description")
                    or obj.get("full_text")
                    or obj.get("text")
                    or ""
                )

                if t: title_map[jid] = t
                if c: sector_map[jid] = c
                if d: desc_map[jid] = d

    return title_map, sector_map, desc_map, total_files, found


def load_stage2_npz(in_npz: Path, ctx: str):
    """
    Accepts either:
      - old schema: short_desc, tasks_and_skills
      - cosine_merged schema: role_text, taskskill_text
    Returns canonical arrays: job_ids, job_desc, job_tasks, domains, titles
    """
    with np.load(in_npz, allow_pickle=True) as z:
        require_npz_keys(z, {"job_ids", "domains", "candidates"}, ctx)

        job_ids = z["job_ids"].astype(str)
        domains = z["domains"].astype(object)
        titles = z["candidates"]

        if "short_desc" in z.files:
            job_desc = z["short_desc"].astype(object)
        elif "role_text" in z.files:
            job_desc = z["role_text"].astype(object)
        else:
            raise KeyError(f"{ctx}: missing short_desc/role_text")

        if "tasks_and_skills" in z.files:
            job_tasks = z["tasks_and_skills"].astype(object)
        elif "taskskill_text" in z.files:
            job_tasks = z["taskskill_text"].astype(object)
        else:
            raise KeyError(f"{ctx}: missing tasks_and_skills/taskskill_text")

    return job_ids, job_desc, job_tasks, domains, titles


# ============================================================
# RUN
# ============================================================
print("START:", datetime.now().isoformat(timespec="seconds"), flush=True)
print("DEV_ROOT:", DEV_ROOT, flush=True)
print("JSONL_ROOT:", JSONL_ROOT, flush=True)
print("STAGE3_ROOT:", STAGE3_ROOT, flush=True)

for embed in EMBEDS:
    print("\n" + "=" * 110, flush=True)
    print("EMBED:", embed, flush=True)

    FINAL_DIR = DEV_ROOT / embed
    OUT_DIR = STAGE3_ROOT / embed
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("FINAL_DIR:", FINAL_DIR, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)

    for month in MONTHS:
        print("\n------------------------------", flush=True)
        print("MONTH:", month, flush=True)

        in_npz = FINAL_DIR / f"{month}.npz"
        jsonl_dir = JSONL_ROOT / f"{month}_npz"
        out_npz = OUT_DIR / f"{month}.npz"

        if out_npz.exists() and not OVERWRITE:
            print("SKIP exists:", out_npz.name, flush=True)
            continue

        if not in_npz.exists():
            print("MISS stage2 npz:", in_npz, flush=True)
            continue

        if not jsonl_dir.exists():
            print("MISS jsonl dir:", jsonl_dir, flush=True)
            continue

        ctx = f"{embed} {month} {in_npz}"
        job_ids, job_desc, job_tasks, domains, titles = load_stage2_npz(in_npz, ctx)

        n = len(job_ids)
        job_id_set = set(job_ids)

        print("Rows:", n, flush=True)

        # ---- scan jsonl ----
        print("Scanning JSONL dir:", jsonl_dir, flush=True)
        title_map, sector_map, desc_map, total_files, found = scan_jsonl_dir(jsonl_dir, job_id_set)

        print("JSONL files:", total_files, flush=True)
        print("Matched JSONL rows:", found, flush=True)

        # ---- align back to npz order ----
        job_ad_title = np.empty(n, dtype=object)
        job_sector_category = np.empty(n, dtype=object)
        job_description = np.empty(n, dtype=object)

        miss_meta = 0
        miss_desc = 0

        for i, jid in enumerate(job_ids):
            t = title_map.get(jid, "")
            c = sector_map.get(jid, "")
            d = desc_map.get(jid, "")

            if not t and not c:
                miss_meta += 1
            if not d:
                miss_desc += 1

            job_ad_title[i] = t
            job_sector_category[i] = c
            job_description[i] = d

        # ---- write NPZ for LLM runner (canonical keys) ----
        np.savez_compressed(
            out_npz,
            job_ids=job_ids.astype(object),
            job_desc=job_desc,
            job_tasks=job_tasks,
            domain=domains,
            titles=titles,
            job_ad_title=job_ad_title,
            job_sector_category=job_sector_category,
            job_description=job_description,
        )

        print(
            f"OK wrote {out_npz} rows={n:,} "
            f"missing_title_sector={miss_meta:,} missing_full_desc={miss_desc:,}",
            flush=True,
        )

print("\nEND:", datetime.now().isoformat(timespec="seconds"), flush=True)

## TEST 
## TEST

START: 2026-02-21T22:03:25
DEV_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged
JSONL_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary
STAGE3_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection

EMBED: bge_large
FINAL_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/cosine_merged/bge_large
OUT_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large

------------------------------
MONTH: adzuna_month01
Rows: 2657586
Scanning JSONL dir: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/adzuna_month01_npz
JSONL files: 100
Matched JSONL rows: 2657586
OK wrote /projects/a5u/adu_dev/aisi-economy-

In [ ]:
# BELOW IS OLD ABOVE IS TEST

In [2]:
import numpy as np
import json
from pathlib import Path
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
DEV_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines"
)

JSONL_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_1_LLM_summary"
)

STAGE3_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection"
)

EMBEDS = ["bge_large"]#, "e5_large", "gte_large"
MONTHS = [f"adzuna_month{m:02d}" for m in range(1, 15)]
OVERWRITE = True


# ============================================================
# HELPERS
# ============================================================
def require_npz_keys(z, need, ctx=""):
    have = set(z.files)
    missing = sorted(list(set(need) - have))
    if missing:
        raise KeyError(
            f"NPZ missing keys {missing} | ctx={ctx} | have={sorted(list(have))}"
        )


def scan_jsonl_dir(jsonl_dir: Path, job_id_set: set):
    title_map = {}
    sector_map = {}
    desc_map = {}
    total_files = 0
    found = 0

    for p in sorted(jsonl_dir.glob("*.jsonl")):
        total_files += 1
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue

                jid = (
                    obj.get("job_id")
                    or obj.get("jobid")
                    or obj.get("id")
                    or (obj.get("meta") or {}).get("job_id")
                )
                if jid is None:
                    continue

                jid = str(jid)
                if jid not in job_id_set:
                    continue

                found += 1

                t = obj.get("job_ad_title") or obj.get("title") or obj.get("job_title") or ""
                c = obj.get("job_sector_category") or obj.get("category_name") or obj.get("category") or ""
                d = (
                    obj.get("job_description")
                    or obj.get("description")
                    or obj.get("full_text")
                    or obj.get("text")
                    or ""
                )

                if t and jid not in title_map:
                    title_map[jid] = t
                if c and jid not in sector_map:
                    sector_map[jid] = c
                if d and jid not in desc_map:
                    desc_map[jid] = d

    return title_map, sector_map, desc_map, total_files, found


def load_stage2_npz(in_npz: Path, ctx: str):
    """
    Accepts either:
      - old schema: short_desc, tasks_and_skills
      - cosine_merged schema: role_text, taskskill_text
    Returns canonical arrays: job_ids, job_desc, job_tasks, domains, titles
    """
    with np.load(in_npz, allow_pickle=True) as z:
        require_npz_keys(z, {"job_ids", "domains", "candidates"}, ctx)

        job_ids = z["job_ids"].astype(str)
        domains = z["domains"].astype(object)
        titles = z["candidates"]

        if "short_desc" in z.files:
            job_desc = z["short_desc"].astype(object)
        elif "role_text" in z.files:
            job_desc = z["role_text"].astype(object)
        else:
            raise KeyError(f"{ctx}: missing short_desc/role_text")

        if "tasks_and_skills" in z.files:
            job_tasks = z["tasks_and_skills"].astype(object)
        elif "taskskill_text" in z.files:
            job_tasks = z["taskskill_text"].astype(object)
        else:
            raise KeyError(f"{ctx}: missing tasks_and_skills/taskskill_text")

    return job_ids, job_desc, job_tasks, domains, titles


# ============================================================
# RUN
# ============================================================
print("START:", datetime.now().isoformat(timespec="seconds"), flush=True)
print("DEV_ROOT:", DEV_ROOT, flush=True)
print("JSONL_ROOT:", JSONL_ROOT, flush=True)
print("STAGE3_ROOT:", STAGE3_ROOT, flush=True)

for embed in EMBEDS:
    print("\n" + "=" * 110, flush=True)
    print("EMBED:", embed, flush=True)

    FINAL_DIR = DEV_ROOT / embed
    OUT_DIR = STAGE3_ROOT / embed
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("FINAL_DIR:", FINAL_DIR, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)

    for month in MONTHS:
        print("\n------------------------------", flush=True)
        print("MONTH:", month, flush=True)

        in_npz = FINAL_DIR / f"{month}.npz"
        jsonl_dir = JSONL_ROOT / f"{month}_npz"
        out_npz = OUT_DIR / f"{month}.npz"

        if out_npz.exists() and not OVERWRITE:
            print("SKIP exists:", out_npz.name, flush=True)
            continue

        if not in_npz.exists():
            print("MISS stage2 npz:", in_npz, flush=True)
            continue

        if not jsonl_dir.exists():
            print("MISS jsonl dir:", jsonl_dir, flush=True)
            continue

        ctx = f"{embed} {month} {in_npz}"
        job_ids, job_desc, job_tasks, domains, titles = load_stage2_npz(in_npz, ctx)

        n = len(job_ids)
        job_id_set = set(job_ids)

        print("Rows:", n, flush=True)

        # ---- scan jsonl ----
        print("Scanning JSONL dir:", jsonl_dir, flush=True)
        title_map, sector_map, desc_map, total_files, found = scan_jsonl_dir(jsonl_dir, job_id_set)

        print("JSONL files:", total_files, flush=True)
        print("Matched JSONL rows:", found, flush=True)

        # ---- align back to npz order ----
        job_ad_title = np.empty(n, dtype=object)
        job_sector_category = np.empty(n, dtype=object)
        job_description = np.empty(n, dtype=object)

        miss_meta = 0
        miss_desc = 0

        for i, jid in enumerate(job_ids):
            t = title_map.get(jid, "")
            c = sector_map.get(jid, "")
            d = desc_map.get(jid, "")

            if not t and not c:
                miss_meta += 1
            if not d:
                miss_desc += 1

            job_ad_title[i] = t
            job_sector_category[i] = c
            job_description[i] = d

        # ---- write NPZ for LLM runner (canonical keys) ----
        np.savez_compressed(
            out_npz,
            job_ids=job_ids.astype(object),
            job_desc=job_desc,
            job_tasks=job_tasks,
            domain=domains,
            titles=titles,
            job_ad_title=job_ad_title,
            job_sector_category=job_sector_category,
            job_description=job_description,
        )

        print(
            f"OK wrote {out_npz} rows={n:,} "
            f"missing_title_sector={miss_meta:,} missing_full_desc={miss_desc:,}",
            flush=True,
        )

print("\nEND:", datetime.now().isoformat(timespec="seconds"), flush=True)


START: 2026-02-21T21:27:50
DEV_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines
JSONL_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary
STAGE3_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection

EMBED: bge_large
FINAL_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/bge_large
OUT_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large

------------------------------
MONTH: adzuna_month01
MISS stage2 npz: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/cosines/bge_large/adzuna_month01.npz

------------------------------
MONTH: adzuna_month02
MISS stage2 npz: /projects/a5u/adu_dev/aisi-ec

In [1]:
import numpy as np
import json
import os
from pathlib import Path

# --- Configuration ---
PROJECT = "/projects/a5u/adu_dev/aisi-economy-index"
# We use one model folder to get the counts for all months
SOURCE_DIR = Path(PROJECT) / "aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large"
OUTPUT_PATH = Path(PROJECT) / "aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/metadata.json"

def create_metadata():
    metadata = {}
    
    # Get all .npz files in the directory
    files = sorted(list(SOURCE_DIR.glob("adzuna_month*.npz")))
    
    if not files:
        print(f"Error: No .npz files found in {SOURCE_DIR}")
        return

    print(f"Scanning {len(files)} files...")

    for f in files:
        month_key = f.stem  # e.g., 'adzuna_month14'
        try:
            data = np.load(f)
            # Find the first array key to get the length
            first_key = list(data.keys())[0]
            row_count = len(data[first_key])
            metadata[month_key] = row_count
            print(f"  {month_key}: {row_count} rows")
        except Exception as e:
            print(f"  Error reading {f.name}: {e}")

    # Save to JSON
    with open(OUTPUT_PATH, 'w') as jf:
        json.dump(metadata, jf, indent=4)
    
    print(f"\nSuccess! Metadata saved to: {OUTPUT_PATH}")

if __name__ == "__main__":
    create_metadata()

Scanning 14 files...
  Error reading adzuna_month01.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month02.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month03.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month04.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month05.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month06.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month07.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month08.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month09.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month10.npz: Object arrays cannot be loaded when allow_pickle=False
  Error reading adzuna_month11.npz: Object arrays cannot be loaded wh

In [2]:
import numpy as np
import json
import os
from pathlib import Path

# --- Configuration ---
PROJECT = "/projects/a5u/adu_dev/aisi-economy-index"
SOURCE_DIR = Path(PROJECT) / "aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large"
OUTPUT_PATH = Path(PROJECT) / "aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/metadata.json"

def create_metadata():
    metadata = {}
    files = sorted(list(SOURCE_DIR.glob("adzuna_month*.npz")))
    
    if not files:
        print(f"Error: No .npz files found in {SOURCE_DIR}")
        return

    print(f"Scanning {len(files)} files with pickle allowed...")

    for f in files:
        month_key = f.stem
        try:
            # FIX: Added allow_pickle=True here
            data = np.load(f, allow_pickle=True)
            
            first_key = list(data.keys())[0]
            row_count = len(data[first_key])
            metadata[month_key] = row_count
            print(f"  {month_key}: {row_count} rows")
        except Exception as e:
            print(f"  Error reading {f.name}: {e}")

    with open(OUTPUT_PATH, 'w') as jf:
        json.dump(metadata, jf, indent=4)
    
    print(f"\nSuccess! Metadata saved to: {OUTPUT_PATH}")

if __name__ == "__main__":
    create_metadata()

Scanning 14 files with pickle allowed...
  adzuna_month01: 2657586 rows
  adzuna_month02: 1539063 rows
  adzuna_month03: 1684605 rows
  adzuna_month04: 1500944 rows
  adzuna_month05: 1845262 rows
  adzuna_month06: 1287469 rows
  adzuna_month07: 1484251 rows
  adzuna_month08: 1296837 rows
  adzuna_month09: 1269039 rows
  adzuna_month10: 1486281 rows
  adzuna_month11: 1419300 rows
  adzuna_month12: 982172 rows
  adzuna_month13: 1220262 rows
  adzuna_month14: 8426 rows

Success! Metadata saved to: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/metadata.json
